In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "eaf6de0d",
   "metadata": {},
   "source": [
    "# 👥 Module 7, 8 & 9: User-Based CF, Ranking, and the Cold-Start Problem\n",
    "**Movie Recommendation System Using Machine Learning**\n",
    "\n",
    "---\n",
    "\n",
    "### 🎯 Objectives\n",
    "1. Understand **User-to-User Collaborative Filtering** (finding peer neighbors).\n",
    "2. Compare Item-Based CF vs User-Based CF (strengths, weaknesses, catalog coverage).\n",
    "3. Understand the **Cold-Start Problem** (new users with zero ratings).\n",
    "4. Implement the **IMDb Bayesian Weighted Rating** fallback strictly on training data.\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "b788c199",
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import sys\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "\n",
    "sys.path.insert(0, \"..\")\n",
    "from src.recommender import MovieRecommender\n",
    "\n",
    "model = MovieRecommender.load(\"../models/recommender_model.joblib\")\n",
    "print(\"Model ready!\")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "74fe7c88",
   "metadata": {},
   "source": [
    "## 1. Finding Similar Users (Nearest Neighbors)\n",
    "We compute similarity across users based on their rating overlap:\n",
    "$$\\text{sim}(u, v) = \\frac{\\mathbf{r}_u \\cdot \\mathbf{r}_v}{\\|\\mathbf{r}_u\\| \\|\\mathbf{r}_v\\|}$$\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "4b3df73f",
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.similarity import get_top_similar_users\n",
    "\n",
    "target_user = 1\n",
    "similar_users = get_top_similar_users(\n",
    "    user_id=target_user,\n",
    "    user_similarity=model.user_similarity,\n",
    "    user_to_idx=model.user_to_idx,\n",
    "    idx_to_user=model.idx_to_user,\n",
    "    top_k=5\n",
    ")\n",
    "\n",
    "print(f\"Top 5 users most similar to User {target_user}:\")\n",
    "for u, s in similar_users:\n",
    "    print(f\"  • User {u}: Similarity = {s:.4f}\")\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "066d4f83",
   "metadata": {},
   "source": [
    "## 2. Generating User-Based Recommendations"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "e08549f7",
   "metadata": {},
   "outputs": [],
   "source": [
    "user_recs = model.recommend_user_based(user_id=1, top_k=5)\n",
    "display(pd.DataFrame(user_recs))\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "05b95930",
   "metadata": {},
   "source": [
    "## 3. The Cold-Start Problem & Bayesian Popularity Fallback\n",
    "\n",
    "### What is the Cold-Start Problem?\n",
    "When a new user joins, they have rated zero movies.\n",
    "The similarity dot products are zero, so collaborative filtering fails!\n",
    "\n",
    "### Solution: IMDb Bayesian Weighted Rating (WR)\n",
    "$$\\text{WR} = \\left(\\frac{v}{v + m}\\right) R + \\left(\\frac{m}{v + m}\\right) C$$\n",
    "- $v$: Number of ratings the movie received in training data\n",
    "- $m$: Minimum ratings threshold (confidence parameter)\n",
    "- $R$: Average rating of the movie\n",
    "- $C$: Mean rating across the entire training catalog\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "775727b9",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Test brand new user with ID 88888\n",
    "new_user_id = 88888\n",
    "cold_start_recs = model.recommend(user_id=new_user_id, top_k=5)\n",
    "print(f\"Recommendations for Cold-Start User {new_user_id}:\")\n",
    "display(pd.DataFrame(cold_start_recs))\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f479b63e",
   "metadata": {},
   "source": [
    "## 4. Beginner Exercises\n",
    "1. Compare the top 5 recommendations for User 1 between Item-Based CF and User-Based CF. Do they overlap?\n",
    "2. Why is the Bayesian Weighted Rating superior to simple average rating for ranking popular movies?\n"
   ]
  }
 ],
 "metadata": {},
 "nbformat": 4,
 "nbformat_minor": 5
}